# 전처리 결과 확인

원본 CSV → 클리닝(+고상관 제거+indicator) → 이상치 처리 → 메타피처 → die→unit 집계 결과를 확인하는 노트북

## 1. 환경 설정 및 데이터 로드

In [1]:
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    if not os.path.exists("/content/project/2_preprocessing/cleaning.py"):
        os.system("gdown --id 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip")
        os.system("unzip -qo /content/preprocessing.zip -d /content/project")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../setup.py

# ─── 이 노트북에서 사용하는 import ───
from utils.config import PROJECT_ROOT
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈 import
sys.path.insert(0, os.path.join(PROJECT_ROOT, "2_preprocessing"))
from cleaning import run_cleaning
from outlier import run_outlier_treatment
from meta_features import run_meta_features
from aggregation import run_aggregation

# 데이터 로드
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
ys_train = ys["train"]

setup 완료
Xs: (174980, 1091)  |  Ys: train=26,247, val=8,749, test=8,749


## 2. 클리닝 (상수/고결측/중복/고상관 제거 + 결측 imputation + indicator)

In [2]:
xs_train, xs_val, xs_test, clean_cols, clean_report = run_cleaning(
    xs, feat_cols, xs_dict,
    const_threshold=1e-6,      # std가 이 값 이하인 feature 제거
    missing_threshold=0.5,     # 결측률이 이 비율 이상인 feature 제거
    remove_duplicates=True,    # 값이 완전히 동일한 중복 컬럼 제거
    corr_threshold=0.95,       # |r|>0.95 고상관 쌍에서 한쪽 제거
    add_indicator=True,        # 결측률 1%+ feature에 _missing indicator 추가
    indicator_threshold=0.01,
)

클리닝 파이프라인 시작
원본 feature 수: 1087
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 982개
    컬럼: 1087 → 982 (105개 제거)
    DataFrame: (104988, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 977개
    컬럼: 982 → 977 (5개 제거)
    DataFrame: (104988, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 26개, 잔여: 951개
    컬럼: 977 → 951 (26개 제거)
    DataFrame: (104988, 955)

[고상관 제거] threshold=0.95
  제거: 213개, 잔여: 738개
    컬럼: 951 → 738 (213개 제거)
    DataFrame: (104988, 742)

[결측 indicator] 11개 컬럼 추가 (결측률 >= 1%)
[결측 imputation] train median 기준
  imputation 후 잔여 결측: 0

클리닝 완료: 1087 → 738 features (349개 제거)
  + indicator 컬럼: 11개 → 총 749개
  train: (104988, 753)
  val:   (34996, 753)
  test:  (34996, 753)


## 3. 이상치 처리 (Winsorization)

In [3]:
xs_train, xs_val, xs_test, outlier_report = run_outlier_treatment(
    xs_train, xs_val, xs_test, clean_cols,
    lower_pct=0.01,   # 하위 분위수 경계 (이 미만 값을 클리핑)
    upper_pct=0.99,   # 상위 분위수 경계 (이 초과 값을 클리핑)
)

이상치 처리 파이프라인 시작
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 130개
  이상치 > 10%: 65개
[Winsorization] lower=1%, upper=99%
  적용 feature: 749개
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 129개
  이상치 > 10%: 65개

이상치 처리 완료
  이상치 >5%  feature: 130 → 129 (1개 감소)
  이상치 >10% feature: 65 → 65 (0개 감소)
  train: (104988, 753)


## 3.5 메타 피처 생성 (로트 통계 + 웨이퍼 패턴 + die 좌표)

In [4]:
wt_feat_cols = [c for c in clean_cols if not c.endswith("_missing")]

xs_train, xs_val, xs_test, meta_cols = run_meta_features(
    xs_train, xs_val, xs_test,
    feat_cols=wt_feat_cols,
    ys_train=ys_train,
    lot_stats=True,
    lot_agg_funcs=["mean", "std"],
    wafer_pattern=True,
    die_coords=True,
)

all_feat_cols = clean_cols + meta_cols
print(f"\n집계 대상 총 피처: {len(all_feat_cols)}개 "
      f"(WT: {len(wt_feat_cols)}, indicator: {len(clean_cols)-len(wt_feat_cols)}, "
      f"meta: {len(meta_cols)})")

메타 피처 생성 시작
[run_wf_xy 파싱] lot: 28개, wafer: 25개, die_x: 12~66, die_y: 11~32
[run_wf_xy 파싱] lot: 28개, wafer: 25개, die_x: 13~66, die_y: 11~32
[run_wf_xy 파싱] lot: 28개, wafer: 25개, die_x: 12~66, die_y: 11~32
[로트 메타 피처] 1476개 컬럼 추가 (feat × agg = 738 × 2)
  train lot: 28개, val lot: 28개, test lot: 28개

[웨이퍼 패턴] train 웨이퍼 432장 분류:
  Edge: 219장 (50.7%)
  Random: 189장 (43.8%)
  None: 24장 (5.6%)
  One-Hot 컬럼: ['wf_pattern_Edge', 'wf_pattern_None', 'wf_pattern_Random']

[die 좌표 피처] ['die_x', 'die_y'] 확인 완료

메타 피처 생성 완료: 1481개 컬럼 추가
  train: (104988, 2234)
  val:   (34996, 2234)
  test:  (34996, 2234)

집계 대상 총 피처: 2230개 (WT: 738, indicator: 11, meta: 1481)


## 4. Die → Unit 집계

In [5]:
unit_train, unit_val, unit_test, unit_feat_cols = run_aggregation(
    xs_train, xs_val, xs_test, all_feat_cols,
    agg_funcs=None,                             # 기본값: mean+std+min+max+range+median
    use_position_pivot=False,                   # True면 position별 피벗 feature도 추가
    save_csv=True,                              # 집계 결과를 CSV로 저장
)

Die → Unit 집계 시작
  agg_funcs: ['mean', 'std', 'min', 'max', 'range', 'median']
  position_pivot: False
집계 완료: 26,247 units × 13,380 features (agg: ['mean', 'std', 'min', 'max', 'range', 'median'])
집계 완료: 8,749 units × 13,380 features (agg: ['mean', 'std', 'min', 'max', 'range', 'median'])
집계 완료: 8,749 units × 13,380 features (agg: ['mean', 'std', 'min', 'max', 'range', 'median'])

집계 결과:
  train: (26247, 13380)
  val:   (8749, 13380)
  test:  (8749, 13380)
  feature 수: 13380


KeyboardInterrupt: 